# Stage 3 — LLM Explanation Generation

**Reads:** `outputs/ontology_results.json`  
**Writes:** `outputs/explanations.json`

Generates a natural-language, user-adaptive explanation for each text using
the Qwen LLM, grounded in the ontology triples from Stage 2.

Each record saved = Stage 2 record + `"explanation": "..."`

> **Tip:** You can test a single explanation in Section 6 before running the
> full batch. The LLM is only loaded once.

## 1. Imports

In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path(r"C:/Users/vimal/OneDrive/Documents/Uni/BTP/User-Adaptive-XAI/MCC_contrained_decoding")
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from config import (
    EXPLANATIONS_PATH,
    ONTOLOGY_RESULTS_PATH,
    NUM_BEAMS,
    USE_CONSTRAINED_DECODING,
    LAMBDA_MAP,
    USER_CATEGORY,
)
from constrained_decoding import ReadabilityBeamGenerator
from model_loaders import load_llm
from pipeline_helpers import (
    SYSTEM_PROMPT,
    build_prompt,
    checkpoint_exists,
    generate_explanation,
    load_checkpoint,
    save_checkpoint,
)

In [2]:
pipeline_helpers_path = NOTEBOOK_DIR / "pipeline_helpers.py"

print(pipeline_helpers_path.parent)
print("\nContents:")
for item in pipeline_helpers_path.parent.iterdir():
    print(item.name)

C:\Users\vimal\OneDrive\Documents\Uni\BTP\User-Adaptive-XAI\MCC_contrained_decoding

Contents:
00_dataset_extraction.ipynb
01_lime.ipynb
02_ontology.ipynb
03_llm.ipynb
04_analysis.ipynb
05_comparative_analysis.ipynb
config.py
constrained_decoding.py
fix_comparative_analysis.md
mcc_directory_structure.md
model_loaders.py
ontology_helpers.py
outputs
pipeline_helpers.py
test_data.txt
__pycache__


## 2. Configuration

In [3]:
# Set True to re-run even if explanations.json already exists
FORCE_RERUN = True

## 3. Checkpoint check

In [4]:
if checkpoint_exists(EXPLANATIONS_PATH) and not FORCE_RERUN:
    print(f"⚠️  Checkpoint found at '{EXPLANATIONS_PATH}'.")
    print("    Set FORCE_RERUN = True to overwrite.")
    print("    Loading existing results …")
    results = load_checkpoint(EXPLANATIONS_PATH)
else:
    results = None
    print("No checkpoint found (or FORCE_RERUN=True). Will generate explanations.")

No checkpoint found (or FORCE_RERUN=True). Will generate explanations.


## 4. Load Stage 2 output

In [5]:
if results is None:
    if not ONTOLOGY_RESULTS_PATH.exists():
        raise FileNotFoundError(
            f"Ontology results not found at '{ONTOLOGY_RESULTS_PATH}'.\n"
            "Please run 02_ontology.ipynb first."
        )
    ontology_data = load_checkpoint(ONTOLOGY_RESULTS_PATH)
    print(f"Loaded {len(ontology_data)} texts from Stage 2.")

[Checkpoint] Loaded 50 records ← 'outputs\or_exp.json'
Loaded 50 texts from Stage 2.


## 5. Load LLM

In [6]:
if results is None:
    llm_tokenizer, llm_model = load_llm()
    generator = None
    if USE_CONSTRAINED_DECODING:
        generator = ReadabilityBeamGenerator(
            model=llm_model,
            tokenizer=llm_tokenizer,
            num_beams=NUM_BEAMS,
        )
        print(
            f"Constrained decoding enabled | beams={NUM_BEAMS} | ",
            f"lambda(default={USER_CATEGORY})={LAMBDA_MAP.get(USER_CATEGORY, LAMBDA_MAP['EXPERT'])}",
        )
else:
    generator = None

[Loader] Loading LLM 'Qwen/Qwen2.5-1.5B-Instruct' …


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[Loader] LLM ready.

Constrained decoding enabled | beams=4 |  lambda(default=EXPERT)=0.05


## 6. [Optional] Preview prompt for a single sample

Run this cell to inspect what the LLM will receive before running the full batch.

In [7]:
if results is None:
    sample = ontology_data[0]          # ← change index to preview a different text
    prompt = build_prompt(
        text = sample["text"],
        predicted_class=sample["predicted_class"],
        feature_data=sample["feature_data"],
        user_category=sample["user_category"],
    )
    print("── SYSTEM PROMPT ─────────────────────────────────────────")
    print(SYSTEM_PROMPT)
    print("\n── USER PROMPT ───────────────────────────────────────────")
    print(prompt)

── SYSTEM PROMPT ─────────────────────────────────────────
You are a biomedical explanation assistant. Your job is to generate clear, accurate natural language explanations of why a machine learning model made a specific biomedical prediction. You are given:
- The model's predicted class
- Key tokens identified by LIME (local feature attribution) as influential in the prediction
- Ontology-derived ancestor chains for each token, showing its place in the biomedical concept hierarchy
 
Your explanations must be grounded strictly in the provided features and ontology context. Do not introduce facts, diseases, or concepts not present in the input.
 
Adapt your explanation style based on the user category:
- BEGINNER: Use plain, everyday language. Avoid technical jargon. Explain medical terms when they appear. Keep sentences short. The goal is comprehension, not completeness.
- INTERMEDIATE: Balance accessibility with domain accuracy. Define specialized terms briefly. Use medical vocabulary

## 7. Generate explanations (full batch)

In [8]:
if results is None:
    results = []
    for i, item in enumerate(ontology_data):
        print(f"\nGenerating explanation {i + 1}/{len(ontology_data)} …")
        print(f"  Class    : {item['predicted_class']}")
        print(f"  User     : {item['user_category']}")
        print(f"  Features : {[f['feature_word'] for f in item['feature_data']]}")
        if USE_CONSTRAINED_DECODING:
            lam = LAMBDA_MAP.get(item["user_category"], LAMBDA_MAP["EXPERT"])
            print(f"  Lambda   : {lam}")

        explanation = generate_explanation(
            text = item["text"],
            predicted_class=item["predicted_class"],
            feature_data=item["feature_data"],
            user_category=item["user_category"],
            tokenizer=llm_tokenizer,
            model=llm_model,
            generator=generator,
        )

        print(f"  Output   : {explanation[:120]}…")
        results.append({**item, "explanation": explanation})

    print("\n✅ Explanation generation complete.")


Generating explanation 1/50 …
  Class    : Digestive system diseases
  User     : EXPERT
  Features : ['artery']
  Lambda   : 0.05


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Output   : The model classified the biomedical abstract as "Digestive System Diseases" due to the presence of key words and phrases…

Generating explanation 2/50 …
  Class    : Cardiovascular diseases
  User     : EXPERT
  Features : ['cavernous sinus']
  Lambda   : 0.05
  Output   : The model classified the abstract as "Cardiovascular diseases" due to the presence of key terms related to blood vessels…

Generating explanation 3/50 …
  Class    : Digestive system diseases
  User     : EXPERT
  Features : ['ascites', 'peritoneum', 'Endometriosis']
  Lambda   : 0.05
  Output   : The model classified the abstract as a digestive system disease due to the presence of ascites, which is a symptom of th…

Generating explanation 4/50 …
  Class    : Cardiovascular diseases
  User     : EXPERT
  Features : ['claudication']
  Lambda   : 0.05
  Output   : The model classified the abstract as "Cardiovascular diseases" due to the mention of "intermittent claudicatio" and "per…

Generating explanati

## 8. Inspect explanations

In [9]:
for i, r in enumerate(results):
    print(f"\n── Text {i + 1} ──────────────────────────")
    print(f"Class   : {r['predicted_class']}")
    print(f"User    : {r['user_category']}")
    print(f"\nExplanation:\n{r['explanation']}")
    print()


── Text 1 ──────────────────────────
Class   : Digestive system diseases
User    : EXPERT

Explanation:
The model classified the biomedical abstract as "Digestive System Diseases" due to the presence of key words and phrases related to the cardiovascular and respiratory systems. The abstract mentions "cardiac output" and "pulmonary vascular pressures," which are directly associated with the cardiovascular system. Additionally, the mention of "arteries" is crucial because arteries are part of the circulatory system, which plays a vital role in delivering oxygen and nutrients to the body's tissues and removing waste products. The cardiovascular system is closely linked to the digestive system, as both systems work together to maintain overall bodily function. Therefore, the abstract's content is strongly indicative of diseases affecting the cardiovascular or respiratory systems, leading the model to classify it as a digestive system disease.


── Text 2 ──────────────────────────
Class 

## 9. Save checkpoint

In [10]:
save_checkpoint(results, EXPLANATIONS_PATH)
print(f"\n➡️  Continue to notebook 04_analysis.ipynb")

[Checkpoint] Saved 50 records → 'outputs\ex_exp_cd.json'

➡️  Continue to notebook 04_analysis.ipynb
